# Réplica de Ait Hsine y Arabo (2026)
**A privacy-preserving middleware architecture for detecting prompt injection and sensitive data exposure in large-language-model interactions.** *Electronics, 15*(16), 3554. https://doi.org/10.3390/electronics15163554

Este notebook reproduce la evaluación de la sección 5 del artículo:
reglas → heurística → SLM local (Narasimha-0.6B + LoRA) → enrutador de sensibilidad → LLM externo (opcional) → veredicto.

**Antes de empezar:** en Colab, *Entorno de ejecución → Cambiar tipo de entorno → CPU*. El artículo midió la latencia solo en CPU (sección 4.10, p. 13).

## 1. Obtener el código

In [ ]:
# Opción A: desde tu repositorio de GitHub (reemplaza TU_USUARIO)
!git clone https://github.com/TU_USUARIO/replica-aithsine-arabo-2026.git
%cd replica-aithsine-arabo-2026
# Opción B: sube el .zip con el ícono de carpeta y descomprímelo:
# !unzip -q replica-aithsine-arabo-2026.zip && %cd replica-aithsine-arabo-2026

## 2. Instalar dependencias y revisar el entorno

In [ ]:
!pip install -q "transformers>=4.35.0" "peft>=0.13.0" "datasets>=2.18.0" accelerate
import torch, platform
print("GPU disponible:", torch.cuda.is_available(), "(debe ser False para medir latencia como el artículo)")
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h | head -2

## 3. (Opcional) Iniciar sesión en Hugging Face
Solo si algún recurso pide aceptar condiciones. Guarda tu token en *Secretos* de Colab con el nombre `HF_TOKEN`; no lo escribas en el notebook.

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))

## 4. Revisar el conjunto de evaluación
Recurso publicado: `v1adam/Comparison_Dataset` (Data Availability, p. 21). Debe tener 120 prompts: 40 benign (20 generales + 20 hard negatives), 40 jailbreak y 40 sensitive (sección 5, p. 13).
Si no carga, el código usa `data/simulado_120.csv` y lo indica.

In [ ]:
from src.data import load_eval
df, fuente = load_eval("auto")
print(fuente)
print(df.groupby(["label", "subtype"]).size())
df.sample(5, random_state=1)

## 5. Prueba rápida (5 prompts por clase)

In [ ]:
!python run_experiment.py --data auto --limit 5 --out results_prueba

## 6. Experimento completo
Ejecuta reglas, SLM y DeBERTa-v3 sobre los 120 prompts. Con unos 3.2 s por prompt en el SLM (Tabla 5, p. 14), tarda varios minutos en CPU.

Para incluir el LLM externo (Gemini 3 Flash vía OpenRouter, de pago), guarda tu clave en *Secretos* como `OPENROUTER_API_KEY`, descomenta las dos primeras líneas y agrega `--llm`.

In [ ]:
# import os; from google.colab import userdata
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
!python run_experiment.py --data auto --deberta --rules-profile paper --out results

## 7. Comparar con el artículo (Tablas 5 y 8)

In [ ]:
import pandas as pd, json, matplotlib.pyplot as plt
m = pd.read_csv("results/metricas.csv")
cols = ["config", "accuracy", "paper_accuracy", "recall", "paper_recall", "fpr", "paper_fpr", "latency_ms", "paper_latency_ms"]
display(m[[c for c in cols if c in m.columns]].round(4))
print(json.dumps(json.load(open("results/enrutamiento.json")), indent=2, ensure_ascii=False))

ax = m.set_index("config")[["accuracy", "paper_accuracy"]].plot.bar(figsize=(8, 4), rot=15)
ax.set_ylabel("Exactitud"); ax.set_title("Réplica vs. artículo"); ax.legend(["Réplica", "Artículo"])
plt.tight_layout(); plt.savefig("results/replica_vs_articulo.png", dpi=150); plt.show()

## 8. Demostración en vivo del middleware (Fig. 2, p. 10)

In [ ]:
!python demo.py

## 9. Descargar resultados

In [ ]:
!zip -qr resultados.zip results
from google.colab import files
files.download("resultados.zip")